In [1]:
%pip install --upgrade pip
%pip install pandas numpy scikit-learn tensorflow matplotlib seaborn
%pip install --upgrade --force-reinstall narwhals

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached narwhals-2.23.0-py3-none-any.whl.metadata (15 kB)
Using cached narwhals-2.23.0-py3-none-any.whl (458 kB)
  Attempting uninstall: narwhals
    Found existing installation: narwhals 2.23.0
    Uninstalling narwhals-2.23.0:
      Successfully uninstalled narwhals-2.23.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Load data
try:
    train_df = pd.read_csv('../data/p01_bank_data/bank_data_train.csv')
    test_df = pd.read_csv('../data/p01_bank_data/bank_data_test.csv')
    print("Data loaded successfully!")
except FileNotFoundError:
    print("Error: Check file paths.")

TARGET_COL = 'TARGET'
ID_COL = 'ID'

X = train_df.drop(columns=[ID_COL, TARGET_COL])
y = train_df[TARGET_COL]
X_test = test_df.drop(columns=[ID_COL])

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=30))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test_processed = preprocessor.transform(X_test)

print(f"\nNew Memory-Safe Shape for X_train: {X_train.shape}")

Data loaded successfully!


/var/folders/_v/_6fv48812s7dnx3s4tvhd4hr0000gp/T/ipykernel_22336/1975353475.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()



New Memory-Safe Shape for X_train: (284152, 232)


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score

print("="*60)
print("RUNNING NAIVE BASELINE")
print("="*60)
naive_model = DummyClassifier(strategy='most_frequent')
naive_model.fit(X_train, y_train)

naive_auc = roc_auc_score(y_val, naive_model.predict_proba(X_val)[:, 1])
print(f"Naive Baseline AUC: {naive_auc:.4f}\n")

print("="*60)
print("RUNNING LIGHTWEIGHT RANDOM FOREST")
print("="*60)
param_grid = {
    'n_estimators': [100],
    'max_depth': [10, 20],
    'min_samples_split': [5]
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=1)

grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    scoring='roc_auc', 
    cv=3, 
    verbose=1
)

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

rf_auc = roc_auc_score(y_val, best_rf.predict_proba(X_val)[:, 1])
print(f"\nTuned Random Forest AUC: {rf_auc:.4f}")
print("="*60)

RUNNING NAIVE BASELINE
Naive Baseline AUC: 0.5000

RUNNING LIGHTWEIGHT RANDOM FOREST
Fitting 3 folds for each of 2 candidates, totalling 6 fits

Tuned Random Forest AUC: 0.8266


In [5]:
from sklearn.neural_network import MLPClassifier


print("RUNNING NEURAL NETWORK 1: SCIKIT-LEARN MLPCLASSIFIER")


mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=30,
    random_state=42,
    verbose=True
)

print("Training MLPClassifier...")
mlp.fit(X_train, y_train)

mlp_preds_prob = mlp.predict_proba(X_val)[:, 1]
mlp_preds_class = mlp.predict(X_val)

mlp_auc = roc_auc_score(y_val, mlp_preds_prob)
mlp_acc = accuracy_score(y_val, mlp_preds_class)

print(f"\nMLPClassifier Results:")
print(f"-> Accuracy: {mlp_acc:.4f}")
print(f"-> AUC Score: {mlp_auc:.4f}")

RUNNING NEURAL NETWORK 1: SCIKIT-LEARN MLPCLASSIFIER
Training MLPClassifier...
Iteration 1, loss = 0.25458257
Iteration 2, loss = 0.24131389
Iteration 3, loss = 0.23595417
Iteration 4, loss = 0.23236066
Iteration 5, loss = 0.22994285
Iteration 6, loss = 0.22739580
Iteration 7, loss = 0.22575495
Iteration 8, loss = 0.22417090
Iteration 9, loss = 0.22243980
Iteration 10, loss = 0.22137535
Iteration 11, loss = 0.22031347
Iteration 12, loss = 0.21920949
Iteration 13, loss = 0.21784399
Iteration 14, loss = 0.21707856
Iteration 15, loss = 0.21637090
Iteration 16, loss = 0.21507118
Iteration 17, loss = 0.21447336
Iteration 18, loss = 0.21361824
Iteration 19, loss = 0.21278121
Iteration 20, loss = 0.21254154
Iteration 21, loss = 0.21141542
Iteration 22, loss = 0.21104298
Iteration 23, loss = 0.20996684
Iteration 24, loss = 0.20949065
Iteration 25, loss = 0.20931605
Iteration 26, loss = 0.20869316
Iteration 27, loss = 0.20812113
Iteration 28, loss = 0.20767527
Iteration 29, loss = 0.20739616
It

/Users/fouaourikt/Desktop/Slot_Projects/Churn_prediction/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC


print("RUNNING NEURAL NETWORK 2: KERAS (TENSORFLOW)")


input_dim = X_train.shape[1]

keras_model = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    Dense(1, activation='sigmoid')
])

keras_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', AUC(name='auc')]
)

keras_model.summary()

print("\nTraining Keras Model...")
history = keras_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    verbose=1
)


keras_preds_class = (keras_preds_prob >= 0.5).astype(int)

final_keras_auc = roc_auc_score(y_val, keras_preds_prob)
final_keras_acc = accuracy_score(y_val, keras_preds_class)

print(f"\nKeras Model Results:")
print(f"-> Accuracy: {final_keras_acc:.4f}")
print(f"-> AUC Score: {final_keras_auc:.4f}")


RUNNING NEURAL NETWORK 2: KERAS (TENSORFLOW)


/Users/fouaourikt/Desktop/Slot_Projects/Churn_prediction/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        14,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,025 (66.50 KB)

 Trainable params: 17,025 (66.50 KB)

 Non-trainable params: 0 (0.00 B)


Training Keras Model...
Epoch 1/30
4440/4440 ━━━━━━━━━━━━━━━━━━━━ 4s 662us/step - accuracy: 0.9177 - auc: 0.7069 - loss: 0.2644 - val_accuracy: 0.9188 - val_auc: 0.7580 - val_loss: 0.2498
Epoch 2/30
4440/4440 ━━━━━━━━━━━━━━━━━━━━ 3s 603us/step - accuracy: 0.9189 - auc: 0.7578 - loss: 0.2496 - val_accuracy: 0.9189 - val_auc: 0.7787 - val_loss: 0.2431
Epoch 3/30
4440/4440 ━━━━━━━━━━━━━━━━━━━━ 3s 646us/step - accuracy: 0.9190 - auc: 0.7753 - loss: 0.2438 - val_accuracy: 0.9190 - val_auc: 0.7833 - val_loss: 0.2413
Epoch 4/30
4440/4440 ━━━━━━━━━━━━━━━━━━━━ 3s 620us/step - accuracy: 0.9191 - auc: 0.7856 - loss: 0.2405 - val_accuracy: 0.9191 - val_auc: 0.7930 - val_loss: 0.2380
Epoch 5/30
4440/4440 ━━━━━━━━━━━━━━━━━━━━ 3s 576us/step - accuracy: 0.9192 - auc: 0.7933 - loss: 0.2378 - val_accuracy: 0.9192 - val_auc: 0.7983 - val_loss: 0.2362
Epoch 6/30
4440/4440 ━━━━━━━━━━━━━━━━━━━━ 3s 569us/step - accuracy: 0.9193 - auc: 0.7976 - loss: 0.2365 - val_accuracy: 0.9192 - val_auc: 0.8009 - val_loss

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score


print("RUNNING NEURAL NETWORK 4: YOUR CUSTOM OOP NUMPY MODEL")


# 1. YOUR DENSE LAYER CLASS
class DenseLayer:
    def __init__(self, input_dim, output_dim, activation='sigmoid'):
        self.activation = activation
        # He/Xavier initialization
        self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        self.biases = np.zeros((1, output_dim))

        # Adam Parameters
        self.m_b = np.zeros_like(self.biases)
        self.v_b = np.zeros_like(self.biases)
        self.m_W = np.zeros_like(self.weights)
        self.v_W = np.zeros_like(self.weights)
        self.t = 0
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.epsilon = 1e-8
        self.input_data = None
        self.z = None
        self.output = None
    
    def forward(self, input_data):
        self.input_data = input_data
        self.z = np.dot(input_data, self.weights) + self.biases

        if self.activation == 'sigmoid':
            self.output = 1 / (1 + np.exp(-np.clip(self.z, -500, 500)))
        elif self.activation == 'softmax':
            shiftz = self.z - np.max(self.z, axis=1, keepdims=True)
            exps = np.exp(shiftz)
            self.output = exps / np.sum(exps, axis=1, keepdims=True)

        return self.output

    def backward(self, dA, learning_rate):
        m = self.input_data.shape[0]

        # Activation derivative
        if self.activation == 'sigmoid':
            dZ = dA * (self.output * (1 - self.output))
        elif self.activation == 'softmax':
            dZ = dA

        # Gradients
        dW = np.dot(self.input_data.T, dZ) / m
        db = np.sum(dZ, axis=0, keepdims=True) / m

        # Adam Optimizer Update Rules
        self.t += 1
        self.m_W = self.beta1 * self.m_W + (1 - self.beta1) * dW
        self.m_b = self.beta1 * self.m_b + (1 - self.beta1) * db
        
        self.v_W = self.beta2 * self.v_W + (1 - self.beta2) * (dW ** 2)
        self.v_b = self.beta2 * self.v_b + (1 - self.beta2) * (db ** 2)

        m_W_hat = self.m_W / (1 - self.beta1 ** self.t)
        m_b_hat = self.m_b / (1 - self.beta1 ** self.t)

        v_W_hat = self.v_W / (1 - self.beta2 ** self.t)
        v_b_hat = self.v_b / (1 - self.beta2 ** self.t)

        self.weights -= learning_rate * m_W_hat / (np.sqrt(v_W_hat) + self.epsilon)
        self.biases -= learning_rate * m_b_hat / (np.sqrt(v_b_hat) + self.epsilon)

        # Backpropagation step
        dA_prev = np.dot(dZ, self.weights.T)
        return dA_prev

# 2. TRAINING SETUP & HELPER FUNCTIONS
def categorical_cross_entropy(y_true, y_pred):
    exp = 1e-15
    y_pred = np.clip(y_pred, exp, 1 - exp)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

def forward_pass(network, X):
    output = X
    for layer in network:
        output = layer.forward(output)
    return output

# Prepare target labels (Converting binary array 0/1 to one-hot structure for your Softmax layer)
y_train_arr = y_train.values if hasattr(y_train, 'values') else y_train
y_val_arr = y_val.values if hasattr(y_val, 'values') else y_val

y_train_onehot = np.eye(2)[y_train_arr]
y_val_onehot = np.eye(2)[y_val_arr]

# Define network structure using your processed feature size
input_features = X_train.shape[1]
network = [
    DenseLayer(input_dim=input_features, output_dim=64, activation='sigmoid'),
    DenseLayer(input_dim=64, output_dim=32, activation='sigmoid'),
    DenseLayer(input_dim=32, output_dim=16, activation='sigmoid'),
    DenseLayer(input_dim=16, output_dim=2, activation='softmax')
]

print(f"Network initialized using {len(network)} Dense layers with Adam optimization.")

# 3. YOUR MODIFIED TRAINING LOOP
epochs = 150
learning_rate = 0.001

for epoch in range(epochs):
    # Forward
    output = forward_pass(network, X_train)
    
    # Calculate Loss
    log_loss = categorical_cross_entropy(y_train_onehot, output)
    
    # Backward Pass
    dA = output - y_train_onehot
    for layer in reversed(network):
        dA = layer.backward(dA, learning_rate)
        
    if epoch % 20 == 0 or epoch == epochs - 1:
        val_output = forward_pass(network, X_val)
        val_loss = categorical_cross_entropy(y_val_onehot, val_output)
        train_preds_class = np.argmax(output, axis=1)
        val_preds_class = np.argmax(val_output, axis=1)
        val_preds_prob = val_output[:, 1]
        
        train_acc = np.mean(train_preds_class == y_train_arr)
        val_acc = np.mean(val_preds_class == y_val_arr)
        val_auc = roc_auc_score(y_val_arr, val_preds_prob)
        
        print(f"Epoch {epoch:03d}/{epochs} - Loss: {log_loss:.4f} - Val Loss: {val_loss:.4f} - Accuracy: {val_acc:.4f} - Val AUC: {val_auc:.4f}")

final_val_out = forward_pass(network, X_val)
numpy_preds_prob = final_val_out[:, 1]
numpy_preds_class = np.argmax(final_val_out, axis=1)

numpy_auc = roc_auc_score(y_val_arr, numpy_preds_prob)
numpy_acc = accuracy_score(y_val_arr, numpy_preds_class)

print("\nCustom NumPy Neural Network Training Complete!")
print(f"-> Final Accuracy: {numpy_acc:.4f}")
print(f"-> Final AUC Score: {numpy_auc:.4f}")


RUNNING NEURAL NETWORK 4: YOUR CUSTOM OOP NUMPY MODEL
Network initialized using 4 Dense layers with Adam optimization.
Epoch 000/150 - Loss: 0.3095 - Val Loss: 0.3039 - Accuracy: 0.9186 - Val AUC: 0.5342
Epoch 020/150 - Loss: 0.2805 - Val Loss: 0.2806 - Accuracy: 0.9186 - Val AUC: 0.6234
Epoch 040/150 - Loss: 0.2766 - Val Loss: 0.2769 - Accuracy: 0.9186 - Val AUC: 0.6311
Epoch 060/150 - Loss: 0.2739 - Val Loss: 0.2744 - Accuracy: 0.9186 - Val AUC: 0.6414
Epoch 080/150 - Loss: 0.2714 - Val Loss: 0.2720 - Accuracy: 0.9186 - Val AUC: 0.6527
Epoch 100/150 - Loss: 0.2691 - Val Loss: 0.2696 - Accuracy: 0.9186 - Val AUC: 0.6658
Epoch 120/150 - Loss: 0.2668 - Val Loss: 0.2674 - Accuracy: 0.9186 - Val AUC: 0.6788
Epoch 140/150 - Loss: 0.2646 - Val Loss: 0.2652 - Accuracy: 0.9186 - Val AUC: 0.6900
Epoch 149/150 - Loss: 0.2637 - Val Loss: 0.2643 - Accuracy: 0.9186 - Val AUC: 0.6941

Custom NumPy Neural Network Training Complete!
-> Final Accuracy: 0.9186
-> Final AUC Score: 0.6941
